# GenAI Newsletter — Multi‑Agent System (Manager + 4 Agents)

In [14]:
!pip install --quiet tavily-python feedparser openai mcp python-dotenv tiktoken
!pip install --quiet transformers accelerate sentencepiece bitsandbytes
print('[install] If needed, install the packages by uncommenting the lines above.')

[install] If needed, install the packages by uncommenting the lines above.


In [ ]:
import os
import re
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

## Settings / Logging / Utils

In [ ]:
# ---------- Settings / Logging / Utils ----------
os.environ['TAVILY_API_KEY'] = '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'

@dataclass
class Settings:
    llm_provider: str = os.getenv('LLM_PROVIDER', 'huggingface')
    huggingface_model: str = os.getenv('HUGGINGFACE_MODEL', 'Qwen/Qwen2.5-1.5B-Instruct')
    openai_api_key: str = os.getenv('OPENAI_API_KEY', '')
    openai_base_url: str = os.getenv('OPENAI_BASE_URL', 'https://api.openai.com/v1')
    openai_model: str = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
    ollama_host: str = os.getenv('OLLAMA_HOST', 'http://localhost:11434')
    ollama_model: str = os.getenv('OLLAMA_MODEL', 'qwen2.5:1.5b-instruct')
    tavily_api_key: str = os.getenv('TAVILY_API_KEY', '')
    default_temperature: float = float(os.getenv('LLM_TEMPERATURE', '0.4'))
    default_max_tokens: int = int(os.getenv('LLM_MAX_TOKENS', '1500'))
    debug: bool = os.getenv('DEBUG', '0') == '1'

class Log:
    enabled = True
    debug_enabled = False

    @classmethod
    def configure(cls, enabled=True, debug=False):
        cls.enabled = enabled
        cls.debug_enabled = debug

    @classmethod
    def debug(cls, *a):
        if cls.enabled and cls.debug_enabled:
            print('[DEBUG]', *a)

    @classmethod
    def info(cls, *a):
        if cls.enabled:
            print('[INFO]', *a)

    @classmethod
    def warn(cls, *a):
        if cls.enabled:
            print('[WARN]', *a)

class Utils:
    @staticmethod
    def time_window(days: int = 30) -> Tuple[str, str]:
        end = datetime.utcnow()
        start = end - timedelta(days=days)
        return start.isoformat(), end.isoformat()

    @staticmethod
    def clean_text(s: Optional[str]) -> str:
        return re.sub(r'\s+', ' ', (s or '')).strip()

    @staticmethod
    def save_markdown(text: str, filename: str = 'newsletter.md') -> str:
        p = Path(filename).resolve()
        p.write_text(text, encoding='utf-8')
        Log.info('[save] wrote', str(p))
        return str(p)

SETTINGS = Settings()
Log.configure(True, SETTINGS.debug)


## LLM Client

In [ ]:
# ---------- LLM Client ----------
class LLMClient:
    def __init__(self, settings: Settings):
        self.s = settings

    def complete(
        self,
        system_prompt: str,
        user_prompt: str,
        model: Optional[str] = None,
        temperature: Optional[float] = None,
        max_tokens: Optional[int] = None
    ) -> str:
        provider = (self.s.llm_provider or 'huggingface').lower()
        temperature = self.s.default_temperature if temperature is None else float(temperature)
        max_tokens = self.s.default_max_tokens if max_tokens is None else int(max_tokens)
        model = model or (self.s.huggingface_model if provider == 'huggingface' else None)
        if provider == 'huggingface':
            try:
                from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
                import torch
            except Exception:
                return '[MOCKED HF]\n' + user_prompt[:800]
            tok = AutoTokenizer.from_pretrained(model)
            mdl = AutoModelForCausalLM.from_pretrained(
                model,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device_map='auto'
            )
            pipe = pipeline('text-generation', model=mdl, tokenizer=tok, device_map='auto')
            prompt = f'System: {system_prompt}\nUser: {user_prompt}\nAssistant:'
            out = pipe(
                prompt,
                max_new_tokens=max_tokens,
                temperature=temperature,
                do_sample=True,
                top_p=0.9
            )[0]['generated_text']
            return out.split('Assistant:', 1)[-1].strip() or '[EMPTY RESPONSE FROM HF]'
        elif provider == 'openai':
            if not self.s.openai_api_key:
                return '[MOCKED OPENAI]\n' + user_prompt[:800]
            try:
                from openai import OpenAI
                client = OpenAI(api_key=self.s.openai_api_key, base_url=self.s.openai_base_url)
                mdl = model or self.s.openai_model
                resp = client.chat.completions.create(
                    model=mdl,
                    temperature=temperature,
                    max_tokens=max_tokens,
                    messages=[
                        {'role': 'system', 'content': system_prompt},
                        {'role': 'user', 'content': user_prompt}
                    ]
                )
                return resp.choices[0].message.content
            except Exception as e:
                return f'[ERROR OPENAI] {e}'
        else:
            import requests
            mdl = model or self.s.ollama_model
            url = f"{self.s.ollama_host.rstrip('/')}/api/chat"
            payload = {
                'model': mdl,
                'messages': [
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': user_prompt}
                ],
                'stream': False,
                'options': {'temperature': temperature, 'num_predict': max_tokens}
            }
            try:
                r = requests.post(url, json=payload, timeout=120)
                r.raise_for_status()
                data = r.json()
                return (data.get('message', {}) or {}).get('content', '')
            except Exception as e:
                return f'[ERROR OLLAMA] {e}'


## Agents

### Research Agent

In [ ]:
# ---------- Agents ----------
class ResearchAgent:
    def __init__(self, settings: Settings):
        self.s = settings

    def run(
        self,
        blackboard: Dict[str, Any],
        days: int = 30,
        interests: Optional[List[str]] = None,
        max_per_interest: int = 6
    ) -> None:
        items = []
        if interests is None:
            interests = ['LLM', 'multimodal', 'agents', 'open source', 'safety']
        if self.s.tavily_api_key:
            try:
                from tavily import TavilyClient
                tc = TavilyClient(api_key=self.s.tavily_api_key)
                start, _ = Utils.time_window(days)
                for topic in interests:
                    q = f"Generative AI {topic} advancements after:{start[:10]}"
                    Log.info('[Research:Tavily]', q)
                    res = tc.search(q, max_results=max_per_interest)
                    for r in res.get('results', []):
                        items.append({
                            'title': r.get('title'),
                            'url': r.get('url'),
                            'content': r.get('content') or r.get('snippet') or '',
                            'published_at': r.get('published_time') or ''
                        })
            except Exception as e:
                Log.warn('[Research:Tavily] error', e)
        if len(items) < 6:
            try:
                import feedparser
                import time as _t
                feeds = [
                    'https://openai.com/blog/rss.xml',
                    'https://research.google/blog/feed/',
                    'https://stability.ai/blog.rss',
                    'https://ai.googleblog.com/feeds/posts/default',
                    'https://huggingface.co/blog/feed.xml'
                ]
                start_iso, _ = Utils.time_window(days)
                start_ts = datetime.fromisoformat(start_iso).timestamp()
                for url in feeds:
                    Log.info('[Research:RSS] fetch', url)
                    fp = feedparser.parse(url)
                    for e in fp.entries[:30]:
                        pub_ts = None
                        if getattr(e, 'published_parsed', None):
                            pub_ts = _t.mktime(e.published_parsed)
                        elif getattr(e, 'updated_parsed', None):
                            pub_ts = _t.mktime(e.updated_parsed)
                        if pub_ts and pub_ts >= start_ts:
                            items.append({
                                'title': e.get('title', ''),
                                'url': e.get('link', ''),
                                'content': Utils.clean_text(e.get('summary', '')),
                                'published_at': e.get('published', e.get('updated', ''))
                            })
            except Exception as e:
                Log.warn('[Research:RSS] error', e)
        seen = set()
        dedup = []
        for it in items:
            u = it.get('url', '')
            if u and u not in seen:
                seen.add(u)
                dedup.append(it)
        blackboard['raw_items'] = dedup
        Log.info('[Research] unique', len(dedup))


### Curator Agent

In [ ]:
class CuratorAgent:
    def __init__(self, keywords: Optional[List[str]] = None):
        self.keywords = keywords or [
            'benchmark', 'release', 'paper', 'framework', 'dataset',
            'open source', 'capability', 'reasoning', 'agent', 'multimodal'
        ]

    def score(self, item: Dict[str, Any]) -> int:
        c = (item.get('content') or '').lower()
        return len(c) + sum(5 for kw in self.keywords if kw in c)

    def _cluster(self, items: List[Dict[str, Any]], k: int = 4) -> List[Dict[str, Any]]:
        if not items:
            return []
        try:
            from sklearn.feature_extraction.text import TfidfVectorizer
            from sklearn.cluster import KMeans
            texts = [(it.get('title') or '') + ' ' + (it.get('content') or '') for it in items]
            X = TfidfVectorizer(max_features=2048).fit_transform(texts)
            k = min(k, len(items))
            labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
            clusters = {}
            for it, lab in zip(items, labels):
                clusters.setdefault(int(lab), []).append(it)
            sections = []
            for lab, group in clusters.items():
                topic = group[0].get('title', 'Section')
                sections.append({'topic': topic, 'items': group})
            return sections
        except Exception as e:
            Log.warn('[Curator:cluster] fallback:', e)
            buckets = {}
            for it in items:
                key = (it.get('title', 'Untitled').split()[:1] or ['Misc'])[0]
                buckets.setdefault(key, []).append(it)
            return [{'topic': k, 'items': v} for k, v in buckets.items()]

    def run(self, blackboard: Dict[str, Any], top_k: int = 12, num_sections: int = 4) -> None:
        items = blackboard.get('raw_items', [])
        ranked = sorted(items, key=self.score, reverse=True)[:top_k]
        blackboard['ranked_items'] = ranked
        sections = self._cluster(ranked, k=num_sections)
        blackboard['sections'] = sections
        Log.info('[Curator] ranked', len(ranked), '| sections', len(sections))



### Summarizer Agent

In [17]:
class SummarizerAgent:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def run(
        self,
        blackboard: Dict[str, Any],
        persona: str = 'Staff ML engineer',
        tone: str = 'analytical',
        region: str = 'global'
    ) -> None:
        sections = blackboard.get('sections', [])
        drafted = []
        for sec in sections:
            topic = sec.get('topic', 'Updates')
            src = []
            for it in sec.get('items', []):
                title = it.get('title', 'Untitled')
                url = it.get('url', '')
                note = Utils.clean_text(it.get('content', ''))[:240]
                src.append(f"- {title} | {url} | {note}")
            system = 'You are a precise, concise AI editor. Produce short, link-rich bullet points.'
            user = (
                f'Topic: {topic}\nPersona: {persona}\nTone: {tone}\nRegion: {region}\n\n'
                'Sources (title | url | note):\n' + '\n'.join(src) + '\n\n'
                'Write 3-5 *short* bullets. Each bullet must reference at least one source URL inline. Neutral tone.'
            )
            Log.info('[Summarizer] topic ->', topic)
            bullets_text = self.llm.complete(system, user, max_tokens=500)
            bullets = [b.strip('-*• ').strip() for b in bullets_text.split('\n') if b.strip()]
            drafted.append({'topic': topic, 'bullets': bullets})
        blackboard['draft_sections'] = drafted
        Log.info('[Summarizer] drafted', len(drafted))



### Editor Agent

In [ ]:
class EditorAgent:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def run(
        self,
        blackboard: Dict[str, Any],
        persona: str,
        tone: str,
        region: str,
        filename: str
    ) -> None:
        drafted = blackboard.get('draft_sections', [])
        ranked = blackboard.get('ranked_items', [])
        today = datetime.utcnow().strftime('%Y-%m-%d')
        fr = []
        for it in ranked[:5]:
            t = it.get('title', 'Untitled')
            u = it.get('url', '')
            if u:
                fr.append(f'- [{t}]({u})')
        system = 'You are a pragmatic AI chief-of-staff. Write a brief, actionable wrap-up.'
        user = (
            f'Persona: {persona}\nTone: {tone}\nRegion: {region}\n\n'
            'Context: The newsletter covers topics above. In 3-5 sentences, summarize what matters and next actions.'
        )
        wrap = self.llm.complete(system, user, max_tokens=220)
        md = [f'# GenAI Newsletter — {today}', '']
        for sec in drafted:
            md.append(f"## {sec.get('topic', 'Updates')}")
            for b in sec.get('bullets', []):
                md.append('- ' + b if not b.startswith('- ') else b)
            md.append('')
        md.append('---')
        md.append('## What It Means')
        md.append(wrap)
        md.append('')
        if fr:
            md.append('## Further Reading')
        md.extend(fr)
        md.append('')
        path = Utils.save_markdown('\n'.join(md), filename=filename)
        blackboard['newsletter_path'] = path
        Log.info('[Editor] saved ->', path)



### Manager Agent

In [15]:
class ManagerAgent:
    def __init__(self, settings: Settings):
        self.s = settings
        self.llm = LLMClient(settings)
        self.research = ResearchAgent(settings)
        self.curator = CuratorAgent()
        self.summarizer = SummarizerAgent(self.llm)
        self.editor = EditorAgent(self.llm)

    def run(
        self,
        days: int = 30,
        interests: Optional[List[str]] = None,
        persona: str = 'Staff ML engineer',
        tone: str = 'analytical',
        region: str = 'global',
        filename: str = 'newsletter_multiagent.md'
    ) -> Dict[str, Any]:
        blackboard = {}
        Log.info('[Manager] Research')
        self.research.run(blackboard, days=days, interests=interests)
        if not blackboard.get('raw_items'):
            Log.warn('[Manager] No items -> mock')
            blackboard['raw_items'] = [
                {'title': 'Example Multimodal', 'url': 'https://example.com/mm', 'content': 'New multimodal model enhances tool-use and grounding.'},
                {'title': 'Efficient Finetune', 'url': 'https://example.com/ft', 'content': 'Method reduces compute by ~40% for domain adaptation.'},
                {'title': 'Open Source Toolkit', 'url': 'https://example.com/oss', 'content': 'Toolkit for evaluating and tracing agent workflows.'}
            ]
        Log.info('[Manager] Curate')
        self.curator.run(blackboard, top_k=12, num_sections=10)
        Log.info('[Manager] Summarize')
        self.summarizer.run(blackboard, persona=persona, tone=tone, region=region)
        Log.info('[Manager] Edit')
        self.editor.run(blackboard, persona=persona, tone=tone, region=region, filename=filename)
        return {
            'path': blackboard.get('newsletter_path'),
            'sections': blackboard.get('draft_sections'),
            'ranked_items': blackboard.get('ranked_items'),
            'raw_items': blackboard.get('raw_items')
        }

print('[core] Multi-agent classes loaded.')


[core] Multi-agent classes loaded.


## Demo

In [16]:
mgr = ManagerAgent(SETTINGS)
out = mgr.run(days=30, interests=['LLM','agents','multimodal'], filename='newsletter_multiagent_demo.md')
print('[demo] saved at:', out['path'])
print('[demo] sections count:', len(out.get('sections') or []))

[INFO] [Manager] Research
[INFO] [Research:Tavily] Generative AI LLM advancements after:2025-09-16


/tmp/ipython-input-1604044147.py:52: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


[INFO] [Research:Tavily] Generative AI agents advancements after:2025-09-16
[INFO] [Research:Tavily] Generative AI multimodal advancements after:2025-09-16
[INFO] [Research] unique 15
[INFO] [Manager] Curate
[INFO] [Curator] ranked 12 | sections 10
[INFO] [Manager] Summarize
[INFO] [Summarizer] topic -> Contemporary AI Trends -AI, Generative AI, LLMs, Agentic ...


Device set to use cuda:0


[INFO] [Summarizer] topic -> The Latest AI News and AI Breakthroughs that Matter Most


...

[INFO] [Summarizer] topic -> Agentic AI vs Generative AI: Key Differences Explained


Device set to use cuda:0


[INFO] [Summarizer] drafted 10
[INFO] [Manager] Edit


/tmp/ipython-input-1604044147.py:316: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow().strftime('%Y-%m-%d')
Device set to use cuda:0


[INFO] [save] wrote /content/newsletter_multiagent_demo.md
[INFO] [Editor] saved -> /content/newsletter_multiagent_demo.md
[demo] saved at: /content/newsletter_multiagent_demo.md
[demo] sections count: 10
